# M03. 타입 변환

> 📌 **언제 필요한가**  
> 숫자가 문자열로 들어와 있어서 계산이 안 될 때.  
> 예: `"1,234"`(문자열) → `1234`(정수), `"45.6"` → `45.6`(실수)

## 이 모듈에서 배울 것

- 컬럼 타입 확인 (`dtypes`)
- 문자열 → 숫자 변환 (`astype`)
- 콤마 제거 + 변환 (`str.replace` + `astype`)
- 변환 실패 처리

---


## 1. 왜 타입 변환이 필요한가

받은 데이터에서 숫자가 문자열로 들어와 있는 경우가 흔해요. 그러면 계산이 안 돼요.


In [1]:
import pandas as pd

# 예시: 콤마가 박힌 숫자
example = pd.DataFrame({
    '도시': ['서울', '부산', '인천'],
    '인구': ['9,500,000', '3,400,000', '2,900,000']  # ← 문자열!
})
print("타입 확인:")
print(example.dtypes)
print()
print(example)


타입 확인:
도시    object
인구    object
dtype: object

   도시         인구
0  서울  9,500,000
1  부산  3,400,000
2  인천  2,900,000


In [2]:
# 그냥 더하면? → 문자열 합치기가 됨
try:
    print(example['인구'].sum())  # 이건 문자열 이어붙이기
except Exception as e:
    print(f"에러: {e}")


9,500,0003,400,0002,900,000


## 2. 콤마 제거 + 정수 변환


In [3]:
# Step 1: 콤마 제거
example['인구'] = example['인구'].str.replace(',', '')

# Step 2: 정수로 변환
example['인구'] = example['인구'].astype(int)

print("변환 후 타입:")
print(example.dtypes)
print()
print(example)
print()
print(f"총합: {example['인구'].sum():,}")  # 이제 계산됨!


변환 후 타입:
도시    object
인구     int64
dtype: object

   도시       인구
0  서울  9500000
1  부산  3400000
2  인천  2900000

총합: 15,800,000


## 3. 한 줄로 — 체이닝


In [4]:
example2 = pd.DataFrame({
    '도시': ['서울', '부산'],
    '인구': ['9,500,000', '3,400,000']
})

# 체이닝
example2['인구'] = example2['인구'].str.replace(',', '').astype(int)
example2


,도시,인구
0,서울,9500000
1,부산,3400000


## 4. 다양한 타입

| 변환 | 코드 |
|---|---|
| → 정수 | `.astype(int)` |
| → 실수 | `.astype(float)` |
| → 문자열 | `.astype(str)` |
| → 카테고리 | `.astype('category')` |
| → 날짜 | `pd.to_datetime(...)` |


In [5]:
# 날짜 변환 예시
dates = pd.DataFrame({
    '날짜': ['2024-01-15', '2024-02-20', '2024-03-10']
})
print("변환 전:", dates['날짜'].dtype)

dates['날짜'] = pd.to_datetime(dates['날짜'])
print("변환 후:", dates['날짜'].dtype)
print()

# 날짜 컬럼은 .dt로 다양한 정보 접근
dates['연도'] = dates['날짜'].dt.year
dates['월'] = dates['날짜'].dt.month
dates['요일'] = dates['날짜'].dt.day_name()
dates


변환 전: object
변환 후: datetime64[ns]



,날짜,연도,월,요일
0,2024-01-15,2024,1,Monday
1,2024-02-20,2024,2,Tuesday
2,2024-03-10,2024,3,Sunday


## 5. 변환 실패 처리

`astype(int)`는 변환 못 하는 값이 있으면 에러 발생.


In [6]:
messy = pd.Series(['100', '200', '오류', '400'])
try:
    messy.astype(int)
except ValueError as e:
    print(f"에러: {e}")


에러: invalid literal for int() with base 10: '오류'


In [7]:
# 해결: pd.to_numeric의 errors='coerce' — 변환 실패 시 NaN
import numpy as np

clean = pd.to_numeric(messy, errors='coerce')
print(clean)
print(f"\n타입: {clean.dtype}")


0    100.0
1    200.0
2      NaN
3    400.0
dtype: float64

타입: float64


`pd.to_numeric` + `errors='coerce'`가 안전한 변환의 표준 방법이에요.


## 6. 본인 데이터에 적용해보기 ✏️


In [8]:
# 본인 데이터의 모든 컬럼 타입 확인
# print(my_df.dtypes)
# 
# # 'object'인데 숫자여야 하는 컬럼 찾기
# # 변환:
# # my_df['숫자컬럼'] = pd.to_numeric(
# #     my_df['숫자컬럼'].str.replace(',', ''),
# #     errors='coerce'
# # )


## 7. ⚠️ 주의사항

### 7.1 NaN이 섞인 경우
`astype(int)`는 NaN을 못 다룸 (int는 NaN 표현 못 함).  
**해결**: 먼저 결측 처리하거나 `Int64` (대문자, nullable int) 사용.

### 7.2 콤마/공백/단위 섞인 숫자
`"1,234 명"` 같은 거. 단계별 처리:
```python
s.str.replace(',', '').str.replace(' 명', '').astype(int)
```

### 7.3 날짜 형식 다양
`'2024-01-15'`, `'2024/1/15'`, `'24년 1월 15일'`...  
**해결**: `pd.to_datetime(s, format='%Y-%m-%d')`로 명시.


## 8. 📚 더 알아보기

- `df.convert_dtypes()` — 자동으로 최적 타입 추론
- `df.infer_objects()` — object 컬럼들 자동 추론
- `pd.api.types.is_numeric_dtype(col)` — 타입 체크
